In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append('../')

import pandas as pd
import os
import subprocess
import zipfile
import matplotlib.pyplot as plt
import lightgbm as lgb
import numpy as np

from itertools import product
from sklearn.model_selection import GroupShuffleSplit
from itertools import chain
from sklearn.model_selection import GroupKFold
from src.utils import * 
from src.feature_engineering import *
from src.pipeline import *
from src.run import *
from src.encoding import *

In [2]:
train_df = basic_data_pipeline()

Train shape: (18145372, 126)
Mem. usage decreased to 13843.82 Mb (19.2% reduction)


In [ ]:
mean_score, std_score, scores = get_cv_score(train_df, n_splits=3)

/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_groups = train_df.groupby('ranker_id').size().values
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_groups = val_df.groupby('ranker_id').size().values # df/series -> numpy
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

Fold 1: Hit Rate@3=0.4666, AUC=0.8121


/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_groups = train_df.groupby('ranker_id').size().values
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_groups = val_df.groupby('ranker_id').size().values # df/series -> numpy
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

Fold 2: Hit Rate@3=0.4662, AUC=0.8040


/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_groups = train_df.groupby('ranker_id').size().values
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_groups = val_df.groupby('ranker_id').size().values # df/series -> numpy
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

Fold 3: Hit Rate@3=0.4654, AUC=0.8056
Mean: 0.4661 ± 0.0005


------------
## Feature Engineering 
------------

- 기본적인 데이터 변환 마무리: frequentFlyer

In [3]:
train_df[['frequentFlyer_count', 'frequentFlyer']].head()

,frequentFlyer_count,frequentFlyer
0,3,S7/SU/UT
1,3,S7/SU/UT
2,3,S7/SU/UT
3,3,S7/SU/UT
4,3,S7/SU/UT


In [4]:
train_df = fe_columns(train_df)

In [18]:
pd.reset_option('display.max_columns')
pd.set_option('display.max_columns', None)
train_df[['frequentFlyer', 'legs0_segments0_operatingCarrier_code', 'legs0_segments0_marketingCarrier_code']].head(10)

,frequentFlyer,legs0_segments0_operatingCarrier_code,legs0_segments0_marketingCarrier_code
0,S7/SU/UT,KV,KV
1,S7/SU/UT,S7,S7
2,S7/SU/UT,S7,S7
3,S7/SU/UT,S7,S7
4,S7/SU/UT,S7,S7
5,S7/SU/UT,S7,S7
6,S7/SU/UT,S7,S7
7,S7/SU/UT,S7,S7
8,S7/SU/UT,S7,S7
9,S7/SU/UT,S7,S7


df.apply(함수, axis=1)  
axis=1: 각 행에 대해 함수 적용, axis=0: 각 열에 대해 함수 적용

In [5]:
train_df[['frequentFlyer', 'legs0_segments0_operatingCarrier_code', 'ff_company_matches']].head(10)

,frequentFlyer,legs0_segments0_operatingCarrier_code,ff_company_matches
0,S7/SU/UT,KV,False
1,S7/SU/UT,S7,True
2,S7/SU/UT,S7,True
3,S7/SU/UT,S7,True
4,S7/SU/UT,S7,True
5,S7/SU/UT,S7,True
6,S7/SU/UT,S7,True
7,S7/SU/UT,S7,True
8,S7/SU/UT,S7,True
9,S7/SU/UT,S7,True


In [23]:
top_company = train_df['legs0_segments0_operatingCarrier_code'].value_counts().head(30)
top_company = top_company > 100000
top_company = top_company[top_company].index.tolist()

In [26]:
for company in top_company:
    train_df[f'is_{company}'] = (
        train_df['legs0_segments0_operatingCarrier_code'] == company).astype(int)

In [30]:
train_test = train_df.drop(columns = 'frequentFlyer')
get_cv_score(train_test)

/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_groups = train_df.groupby('ranker_id').size().values
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_groups = val_df.groupby('ranker_id').size().values # df/series -> numpy
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

Fold 1: Hit Rate@3=0.4666, AUC=0.8068


/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_groups = train_df.groupby('ranker_id').size().values
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_groups = val_df.groupby('ranker_id').size().values # df/series -> numpy
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

Fold 2: Hit Rate@3=0.4661, AUC=0.8175


/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_groups = train_df.groupby('ranker_id').size().values
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_groups = val_df.groupby('ranker_id').size().values # df/series -> numpy
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

...

/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_groups = train_df.groupby('ranker_id').size().values
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_groups = val_df.groupby('ranker_id').size().values # df/series -> numpy
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

Fold 4: Hit Rate@3=0.4661, AUC=0.8061


/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_groups = train_df.groupby('ranker_id').size().values
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_groups = val_df.groupby('ranker_id').size().values # df/series -> numpy
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

Fold 5: Hit Rate@3=0.4659, AUC=0.8157
Mean: 0.4669 ± 0.0014


(np.float64(0.46687556165699445),
 np.float64(0.001431596128247998),
 [0.4666335759982352,
  0.46609002258828713,
  0.46969447644122647,
  0.46608206485771014,
  0.46587766839951333])

In [31]:
get_cv_score(train_test, n_splits=3)

/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_groups = train_df.groupby('ranker_id').size().values
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_groups = val_df.groupby('ranker_id').size().values # df/series -> numpy
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

Fold 1: Hit Rate@3=0.4659, AUC=0.8162


/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_groups = train_df.groupby('ranker_id').size().values
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_groups = val_df.groupby('ranker_id').size().values # df/series -> numpy
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

Fold 2: Hit Rate@3=0.4679, AUC=0.8101


/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:75: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_groups = train_df.groupby('ranker_id').size().values
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:76: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_groups = val_df.groupby('ranker_id').size().values # df/series -> numpy
/Users/jaewoo/Desktop/Kaggle/pred_aeroclub/notebook/../src/run.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

Fold 3: Hit Rate@3=0.4668, AUC=0.8048
Mean: 0.4668 ± 0.0008


(np.float64(0.4668335509974601),
 np.float64(0.000833393136651084),
 [0.4658530938915898, 0.46789021055059477, 0.46675734855019574])

In [37]:
test_needed_cols = train_df.columns.tolist()

In [52]:
will_drop_cols = []
for col in test_needed_cols:
    if 'is_' in col:
        will_drop_cols.append(col)

In [54]:
test_needed_cols = [col for col in test_needed_cols if col not in will_drop_cols]

In [42]:
will_drop_cols = will_drop_cols.append('Id')

In [53]:
will_drop_cols

['is_frequentFlyer',
 'is_FV',
 'is_SU',
 'is_S7',
 'is_U6',
 'is_TK',
 'is_DP',
 'is_UT',
 'is_5N',
 'is_EK',
 'is_N4',
 'is_WZ']

In [57]:
test_needed_cols

['Id',
 'companyID',
 'corporateTariffCode',
 'frequentFlyer',
 'nationality',
 'isAccess3D',
 'isVip',
 'legs0_segments0_aircraft_code',
 'legs0_segments0_arrivalTo_airport_city_iata',
 'legs0_segments0_arrivalTo_airport_iata',
 'legs0_segments0_baggageAllowance_quantity',
 'legs0_segments0_baggageAllowance_weightMeasurementType',
 'legs0_segments0_cabinClass',
 'legs0_segments0_departureFrom_airport_iata',
 'legs0_segments0_flightNumber',
 'legs0_segments0_marketingCarrier_code',
 'legs0_segments0_operatingCarrier_code',
 'legs0_segments0_seatsAvailable',
 'legs0_segments1_aircraft_code',
 'legs0_segments1_arrivalTo_airport_city_iata',
 'legs0_segments1_arrivalTo_airport_iata',
 'legs0_segments1_baggageAllowance_quantity',
 'legs0_segments1_baggageAllowance_weightMeasurementType',
 'legs0_segments1_cabinClass',
 'legs0_segments1_departureFrom_airport_iata',
 'legs0_segments1_flightNumber',
 'legs0_segments1_marketingCarrier_code',
 'legs0_segments1_operatingCarrier_code',
 'legs0_seg

In [62]:
for col in test_needed_cols:
    print(col)

Id
companyID
corporateTariffCode
frequentFlyer
nationality
isAccess3D
isVip
legs0_segments0_aircraft_code
legs0_segments0_arrivalTo_airport_city_iata
legs0_segments0_arrivalTo_airport_iata
legs0_segments0_baggageAllowance_quantity
legs0_segments0_baggageAllowance_weightMeasurementType
legs0_segments0_cabinClass
legs0_segments0_departureFrom_airport_iata
legs0_segments0_flightNumber
legs0_segments0_marketingCarrier_code
legs0_segments0_operatingCarrier_code
legs0_segments0_seatsAvailable
legs0_segments1_aircraft_code
legs0_segments1_arrivalTo_airport_city_iata
legs0_segments1_arrivalTo_airport_iata
legs0_segments1_baggageAllowance_quantity
legs0_segments1_baggageAllowance_weightMeasurementType
legs0_segments1_cabinClass
legs0_segments1_departureFrom_airport_iata
legs0_segments1_flightNumber
legs0_segments1_marketingCarrier_code
legs0_segments1_operatingCarrier_code
legs0_segments1_seatsAvailable
legs0_segments2_aircraft_code
legs0_segments2_arrivalTo_airport_city_iata
legs0_segments2_ar